In [451]:
import numpy as np
import pandas as pd

# sklearn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder

# model
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

# 평가지표
from sklearn.metrics import accuracy_score

In [452]:
train = pd.read_csv('train.csv')
print(train.shape)
train.head(1)

(8693, 14)


,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False


In [453]:
train.describe(include=['object', 'bool'])

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,VIP,Name,Transported
count,8693,8492,8476,8494,8511,8490,8493,8693
unique,8693,3,2,6560,3,2,8473,2
top,0001_01,Earth,False,G/734/S,TRAPPIST-1e,False,Ankalik Nateansive,True
freq,1,4602,5439,8,5915,8291,2,4378


In [454]:
X = train.drop(columns=['Name', 'Transported'])
X.shape

(8693, 12)

In [455]:
y = train['Transported']

In [456]:
# 파생변수 - 탐색보고서 참조 : PassengerId '_'문자열로 분리 후 반환값에 [0]번째만 새로운 'Group' 열로 추가
X['G'] = X['PassengerId'].str.split('_', expand=True)[0]

In [457]:
X.head(2)

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,G
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,0001
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,0002


In [458]:
X[['C1', 'C2', 'C3']] = X['Cabin'].str.split('/', expand=True)
X.head(2)

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,G,C1,C2,C3
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,0001,B,0,P
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,0002,F,0,S


In [459]:
# ISSUE
# X['G']에 숫자형태만 있나? Y
# -> 숫자형으로 변경? N

# X['C2']에 숫자형태만 있나? Y
# -> 숫자형으로 변경? Y
# -> nan값은 있나? Y

In [460]:
print(X['G'].isna().sum())
print(X['C2'].isna().sum())

0
199


In [461]:
X['G'].unique()

array(['0001', '0002', '0003', ..., '9278', '9279', '9280'],
      shape=(6217,), dtype=object)

In [462]:
X['C2'] = X['C2'].astype('Int64') # int 파이썬타입(결측치X), Int64 판다스타입(결측치)

In [463]:
# CryoSleep True/False bool인데 obejct로 되어있다. -> 결측치 'Missing'으로 변경
# 문자로 변경해서 통일
X['CryoSleep'] = X['CryoSleep'].astype('string')
X['CryoSleep'] = X['CryoSleep'].astype('object')

In [464]:
# VIP True/False bool인데 obejct로 되어있다. -> 결측치 'Missing'으로 변경
X['VIP'] = X['VIP'].astype('string')
X['VIP'] = X['VIP'].astype('object')

In [465]:
X.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8693 entries, 0 to 8692
Data columns (total 16 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   PassengerId   8693 non-null   object 
 1   HomePlanet    8492 non-null   object 
 2   CryoSleep     8476 non-null   object 
 3   Cabin         8494 non-null   object 
 4   Destination   8511 non-null   object 
 5   Age           8514 non-null   float64
 6   VIP           8490 non-null   object 
 7   RoomService   8512 non-null   float64
 8   FoodCourt     8510 non-null   float64
 9   ShoppingMall  8485 non-null   float64
 10  Spa           8510 non-null   float64
 11  VRDeck        8505 non-null   float64
 12  G             8693 non-null   object 
 13  C1            8494 non-null   object 
 14  C2            8494 non-null   Int64  
 15  C3            8494 non-null   object 
dtypes: Int64(1), float64(6), object(9)
memory usage: 1.1+ MB


In [466]:
X = X.drop(columns=['PassengerId','Cabin'])

In [467]:
X.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8693 entries, 0 to 8692
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   HomePlanet    8492 non-null   object 
 1   CryoSleep     8476 non-null   object 
 2   Destination   8511 non-null   object 
 3   Age           8514 non-null   float64
 4   VIP           8490 non-null   object 
 5   RoomService   8512 non-null   float64
 6   FoodCourt     8510 non-null   float64
 7   ShoppingMall  8485 non-null   float64
 8   Spa           8510 non-null   float64
 9   VRDeck        8505 non-null   float64
 10  G             8693 non-null   object 
 11  C1            8494 non-null   object 
 12  C2            8494 non-null   Int64  
 13  C3            8494 non-null   object 
dtypes: Int64(1), float64(6), object(7)
memory usage: 959.4+ KB


In [468]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [469]:
X_train.shape

(6954, 14)

In [470]:
cat_cols = X_train.select_dtypes('object').columns
num_cols = X_train.select_dtypes('number').columns
print(cat_cols)
print(num_cols)

Index(['HomePlanet', 'CryoSleep', 'Destination', 'VIP', 'G', 'C1', 'C3'], dtype='object')
Index(['Age', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck',
       'C2'],
      dtype='object')


In [471]:
print(X_train.isna().sum().sum())
print(X_val.isna().sum().sum())

2032
490


In [472]:
# 결측치
# 숫자 -> 각각의 열의 중앙값
# 범주 -> 모든 열에 'Missing'

a = X_train[num_cols].fillna(X_train[num_cols].median())
a2 = X_val[num_cols].fillna(X_val[num_cols].median())

X_train[num_cols] = a
X_val[num_cols] = a2

X_train[cat_cols] = X_train[cat_cols].fillna('Missing')
X_val[cat_cols] = X_val[cat_cols].fillna('Missing')

In [473]:
print(X_train.isna().sum().sum())
print(X_val.isna().sum().sum())

0
0


In [474]:
# 인코딩
encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore').set_output(transform='pandas')

In [475]:
X_train_en = encoder.fit_transform(X_train[cat_cols])
X_val_en = encoder.transform(X_val[cat_cols])

In [476]:
# X_train의 수치형 컬럼 + X_train_en : pd.concat([df1, df2], axix=1)
# X_val의 수치형 컬럼 + X_val_en

In [477]:
X_train_pre = pd.concat([X_train[num_cols], X_train_en], axis=1)
X_val_pre = pd.concat([X_val[num_cols], X_val_en], axis=1)

In [478]:
print(X_train_pre.shape)
print(X_val_pre.shape)

(6954, 5260)
(1739, 5260)


In [479]:
# 학습

In [480]:
# model = RandomForestClassifier(n_estimators=300, random_state=42)
model = LogisticRegression(max_iter=300)

In [481]:
model.fit(X_train_pre, y_train)

/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


LogisticRegression(max_iter=300)

In [482]:
y_pred = model.predict(X_val_pre)

accurracy = accuracy_score(y_val, y_pred)

print(accurracy)

0.78953421506613


RandomForestClassifier : 0.7924094307073031

LogisticRegression(100번) 0.7860839562967222

0.7866589994249569